## Semantic Similarity

In [16]:
import os
import pandas as pd
from scipy.stats import pearsonr
from similarity_scores import mean_wup

def evaluate_similarity(datasets, methods, pre_process=None, suffix=None):
    def col_name(column, suffix):
        return f"{column}_{suffix}" if suffix else column
    
    os.makedirs("data/results", exist_ok=True)
    
    for dataset in datasets:
        data = pd.read_csv(f"data/{dataset}.csv")
        path = f"data/results/{dataset}.csv"
        results = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
        
        s1, s2 = data["S1"], data["S2"]
        
        if pre_process != None:
            s1 = [pre_process(s) for s in s1]
            s2 = [pre_process(s) for s in s2]

        for _, column, method in methods:
            column = col_name(column, suffix)
            if column not in results.columns:
                results[column] = method(s1, s2)
            
        results.to_csv(path, index=False)

        print(
            f"{dataset.upper():<12}"
            + " | ".join(
            f"{name}: {pearsonr(results[col_name(c, suffix)], data["score"])[0]:.4f}"
            for name, c, _ in methods
        ))


datasets = [
    "stss131", "sts12","sts13","sts14","sts15",
    "sts16","sts17","stsb", "sickr", "mrpc"
]

evaluate_similarity(datasets, [("Wu & Palmer", "wup", mean_wup)])

STSS131     Wu & Palmer: 0.3825
STS12       Wu & Palmer: 0.0799
STS13       Wu & Palmer: 0.2712
STS14       Wu & Palmer: 0.2955
STS15       Wu & Palmer: 0.3448
STS16       Wu & Palmer: 0.1946
STS17       Wu & Palmer: 0.1849
STSB        Wu & Palmer: 0.2428
SICKR       Wu & Palmer: 0.4445
MRPC        Wu & Palmer: 0.2065


In [13]:
import gensim.downloader as api

w2v_model = api.load("word2vec-google-news-300")

In [11]:
from similarity_scores import embedding_similarity

methods = [
    ("Wu & Palmer", "wup", mean_wup),
    ("Word2Vec", "w2v", lambda s1, s2: embedding_similarity(w2v_model, s1, s2))
]

evaluate_similarity(datasets, methods)

STSS131     Wu & Palmer: 0.3825 | Word2Vec: 0.7587
STS12       Wu & Palmer: 0.0799 | Word2Vec: 0.5937
STS13       Wu & Palmer: 0.2712 | Word2Vec: 0.6517
STS14       Wu & Palmer: 0.2955 | Word2Vec: 0.6001
STS15       Wu & Palmer: 0.3448 | Word2Vec: 0.6308
STS16       Wu & Palmer: 0.1946 | Word2Vec: 0.5383
STS17       Wu & Palmer: 0.1849 | Word2Vec: 0.6807
STSB        Wu & Palmer: 0.2428 | Word2Vec: 0.6274
SICKR       Wu & Palmer: 0.4445 | Word2Vec: 0.7006
MRPC        Wu & Palmer: 0.2065 | Word2Vec: 0.3961


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import modules

device = "cuda" if torch.cuda.is_available() else "cpu"

def build_model(model_name):
    transformer = modules.Transformer(model_name)

    pooling = modules.Pooling(
        transformer.get_embedding_dimension(),
        pooling_mode="mean"
    )

    return SentenceTransformer(
        modules=[transformer, pooling],
        device=device
    )

ft_model = api.load("fasttext-wiki-news-subwords-300")
albert_model = build_model("albert/albert-base-v2")
roberta_model = build_model("FacebookAI/roberta-base")
sbert_model = SentenceTransformer("paraphrase-mpnet-base-v2", device=device)

In [12]:
from similarity_scores import bert_similarity

methods = [
    ("Wu & Palmer", "wup", mean_wup),
    ("Word2Vec", "w2v", lambda s1, s2: embedding_similarity(w2v_model, s1, s2)),
    ("FastText", "ft", lambda s1, s2: embedding_similarity(ft_model, s1, s2)),
    ("ALBERT", "albert", lambda s1, s2: bert_similarity(albert_model, list(s1), list(s2))),
    ("RoBERTa", "roberta", lambda s1, s2: bert_similarity(roberta_model, list(s1), list(s2))),
    ("SBERT", "sbert", lambda s1, s2: bert_similarity(sbert_model, list(s1), list(s2)))
]

evaluate_similarity(datasets, methods)

STSS131     Wu & Palmer: 0.3825 | Word2Vec: 0.7587 | FastText: 0.4914 | ALBERT: 0.8034 | RoBERTa: 0.7202 | SBERT: 0.9359
STS12       Wu & Palmer: 0.0799 | Word2Vec: 0.5937 | FastText: 0.2017 | ALBERT: 0.3034 | RoBERTa: 0.2577 | SBERT: 0.8168
STS13       Wu & Palmer: 0.2712 | Word2Vec: 0.6517 | FastText: 0.2756 | ALBERT: 0.4902 | RoBERTa: 0.5080 | SBERT: 0.8524
STS14       Wu & Palmer: 0.2955 | Word2Vec: 0.6001 | FastText: 0.2197 | ALBERT: 0.4256 | RoBERTa: 0.4754 | SBERT: 0.8408
STS15       Wu & Palmer: 0.3448 | Word2Vec: 0.6308 | FastText: 0.3102 | ALBERT: 0.5585 | RoBERTa: 0.5528 | SBERT: 0.8610
STS16       Wu & Palmer: 0.1946 | Word2Vec: 0.5383 | FastText: 0.2579 | ALBERT: 0.5427 | RoBERTa: 0.5844 | SBERT: 0.8244
STS17       Wu & Palmer: 0.1849 | Word2Vec: 0.6807 | FastText: 0.2532 | ALBERT: 0.5499 | RoBERTa: 0.6338 | SBERT: 0.8863
STSB        Wu & Palmer: 0.2428 | Word2Vec: 0.6274 | FastText: 0.2740 | ALBERT: 0.5090 | RoBERTa: 0.5438 | SBERT: 0.8711
SICKR       Wu & Palmer: 0.4445 

In [ ]:
from util import lemma_filter, stopword_filter, stop_lemma_filter, alphabet_filter

pre_processes = [
    ("alphabet", alphabet_filter), 
    ("lemma", lemma_filter),
    ("stop", stopword_filter), 
    ("stop_lemma", stop_lemma_filter), 
]

for suffix, process in pre_processes:
    evaluate_similarity(datasets, methods, process, suffix)

In [6]:
import numpy as np
from collections import defaultdict

scores = defaultdict(list)

for dataset in datasets:
    data = pd.read_csv(f"data/{dataset}.csv")
    results = pd.read_csv(f"data/results/{dataset}.csv")
    
    for column in results.columns:
        scores[column].append(pearsonr(results[column], data["score"])[0])

means = {key: np.mean(values) for key, values in scores.items()}

variants = {
    "Default": "", "Alphabet only": "_alphabet", "Lemmatization": "_lemma",
    "Stopword": "_stop", "Lemma & stop": "_stop_lemma",
}

table = pd.DataFrame({
    variant: {name: means[column + suffix] for name, column, _ in methods}
    for variant, suffix in variants.items()
})

print(table.round(4).to_string())

             Default  Alphabet only  Lemmatization  Stopword  Lemma & stop
Wu & Palmer   0.2647         0.2779         0.2447    0.3244        0.3108
Word2Vec      0.6178         0.6520         0.6499    0.6714        0.6809
FastText      0.2985         0.3761         0.3076    0.4387        0.3790
ALBERT        0.5105         0.5031         0.4635    0.4254        0.4015
RoBERTa       0.5336         0.5563         0.5277    0.4731        0.4657
SBERT         0.8193         0.8077         0.8131    0.7882        0.7799


In [ ]:
from itertools import islice
from util import get_verb_adjective, get_synonyms

def ranked_synonyms(word, model):
    if synonyms := get_synonyms(word):
        return word[0], sorted(
            synonyms, 
            key=lambda syn: embedding_similarity(model, [word[0]], [syn]),
            reverse=True
        )
    return None

def candidates(sample):
    for row in sample.itertuples(index=False):
        p1 = get_verb_adjective(row.S1)
        p2 = get_verb_adjective(row.S2)
        
        if not p1 or not p2: continue
        
        synonyms = [ranked_synonyms(w, w2v_model) for p in (p1, p2) for w in p]
        
        if any(result is None for result in synonyms): 
            continue
        
        yield row, synonyms  

def synonym_candidates(file_name):
    df = pd.read_csv(file_name)
    sample = df[["S1", "S2", "score"]].sample(frac=1)
        
    for row, (v1, a1, v2, a2) in islice(candidates(sample), 4):
        print(f'"{row.S1}","{row.S2}",{row.score}')
        print(f"{v1[0]} - {v1[1]}\n{a1[0]} - {a1[1]}")
        print(f"{v2[0]} - {v2[1]}\n{a2[0]} - {a2[1]}\n")
        

synonym_candidates("data/sts17.csv")

In [7]:
evaluate_similarity(["synonyms"], methods)
evaluate_similarity(["negations"], methods)

SYNONYMS    Wu & Palmer: 0.2247 | Word2Vec: 0.5341 | FastText: 0.2818 | ALBERT: 0.5339 | RoBERTa: 0.5675 | SBERT: 0.8852
NEGATIONS   Wu & Palmer: 0.1627 | Word2Vec: 0.4699 | FastText: 0.1434 | ALBERT: 0.4598 | RoBERTa: 0.5067 | SBERT: 0.7042


In [17]:
methods = [
    ("Word2Vec", "w2v", lambda s1, s2: embedding_similarity(w2v_model, s1, s2)),
    ("SBERT", "sbert", lambda s1, s2: bert_similarity(sbert_model, list(s1), list(s2)))
]

evaluate_similarity(["semrel24"], methods)

SEMREL24    Word2Vec: 0.5815 | SBERT: 0.7974
